## Dataset analysis: `student_payments_list15.csv`

Purpose: tuition/fees payment transactions (List15). Used to derive payment trend features per semester.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_payments_list15.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df["PAYMENT_DATE"] = pd.to_datetime(df["PAYMENT_DATE"], errors="coerce")
df["AMOUNT_UGX"] = pd.to_numeric(df["AMOUNT_UGX"], errors="coerce")
df["PAYMENT_STATUS"] = df["PAYMENT_STATUS"].astype(str).str.upper().str.strip()

df.isna().mean().sort_values(ascending=False).head(30)

In [ ]:
# Key uniqueness
df.duplicated(["PAYMENT_ID"]).sum(), df["PAYMENT_ID"].isna().sum()

In [ ]:
df["PAYMENT_STATUS"].value_counts(dropna=False)

In [ ]:
df.groupby("PAYMENT_STATUS").agg(
    txns=("PAYMENT_ID","count"),
    students=("REG_NO","nunique"),
    amount_sum=("AMOUNT_UGX","sum"),
    amount_mean=("AMOUNT_UGX","mean"),
).sort_values("txns", ascending=False)

## Advanced analytics

Focus: payment behavior by semester and how it associates with CGPA (joined to transcript).

In [ ]:
import numpy as np
from analysis_utils import basic_profile, missingness_report, numeric_outlier_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

In [ ]:
# Outlier scan for payment amounts
numeric_outlier_report(df, cols=["AMOUNT_UGX"]).head(20)

In [ ]:
# Aggregate to student-semester and join to transcript CGPA
trans = pd.read_csv(DATA_DIR / "student_transcript_list15.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

p = df.copy()
p["PAYMENT_DATE"] = pd.to_datetime(p["PAYMENT_DATE"], errors="coerce")
p["AMOUNT_UGX"] = pd.to_numeric(p["AMOUNT_UGX"], errors="coerce")
p["PAYMENT_STATUS"] = p["PAYMENT_STATUS"].astype(str).str.upper().str.strip()

p["_success"] = p["PAYMENT_STATUS"].isin(["SUCCESS", "PAID", "COMPLETED"]).astype(int)

agg = p.groupby(["REG_NO", "SEMESTER_INDEX"], as_index=False).agg(
    payment_txn_count=("PAYMENT_ID", "count"),
    payment_success_rate=("_success", "mean"),
    total_paid_success=("AMOUNT_UGX", lambda s: float(s[p.loc[s.index, "_success"].astype(bool)].sum(skipna=True))),
    distinct_methods=("PAYMENT_METHOD", "nunique"),
    distinct_channels=("CHANNEL", "nunique"),
    payment_span_days=("PAYMENT_DATE", lambda s: (s.max() - s.min()).days + 1 if s.notna().any() else np.nan),
)

joined = merge_to_transcript_for_cgpa(agg, trans, on=["REG_NO", "SEMESTER_INDEX"], how="inner")
joined[["CGPA", "payment_txn_count", "payment_success_rate", "total_paid_success"]].corr(numeric_only=True)